# AHP Survey Weights Calculator

This notebook calculates the final aggregated weights from AHP questionnaire responses.

## Process:
1. Load participant responses from CSV
2. Filter for consistent responses (CR ≤ 0.10)
3. Aggregate weights using geometric mean (recommended for AHP)
4. Export final weights for scoring

## Input Format:
CSV with columns: `timestamp`, `user_name`, `consistency_ratio`, `consistency_index`, `lambda_max`, `is_consistent`, `weight_[criterion1]`, `weight_[criterion2]`, ...

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import json
from datetime import datetime

# Configuration
INPUT_CSV = 'responses.csv'  # Path to your responses CSV
OUTPUT_JSON = 'final_weights.json'
OUTPUT_CSV = 'final_weights.csv'
CONSISTENCY_THRESHOLD = 0.10  # Standard AHP threshold

## 1. Load Data

In [ ]:
# Load the CSV file
df = pd.read_csv(INPUT_CSV)

print(f"Total responses loaded: {len(df)}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nFirst few rows:")
df.head()

## 2. Identify Weight Columns

In [ ]:
# Find all weight columns (columns starting with 'weight_')
weight_columns = [col for col in df.columns if col.startswith('weight_')]

# Extract criterion names (remove 'weight_' prefix)
criteria_names = [col.replace('weight_', '') for col in weight_columns]

print(f"Found {len(weight_columns)} criteria:")
for i, (col, name) in enumerate(zip(weight_columns, criteria_names), 1):
    print(f"{i}. {name} (column: {col})")

## 3. Filter Consistent Responses

In [ ]:
# Check if consistency columns exist
if 'consistency_ratio' in df.columns:
    # Filter for consistent responses (CR ≤ threshold)
    df['is_consistent'] = df['consistency_ratio'] <= CONSISTENCY_THRESHOLD
    consistent_df = df[df['is_consistent']].copy()
    
    print(f"Total responses: {len(df)}")
    print(f"Consistent responses (CR ≤ {CONSISTENCY_THRESHOLD}): {len(consistent_df)}")
    print(f"Inconsistent responses: {len(df) - len(consistent_df)}")
    print(f"Consistency rate: {len(consistent_df)/len(df)*100:.1f}%")
    
    # Show consistency ratio distribution
    print(f"\nConsistency Ratio Statistics:")
    print(df['consistency_ratio'].describe())
else:
    print("Warning: No consistency_ratio column found. Using all responses.")
    consistent_df = df.copy()

## 4. Display Individual Responses

In [ ]:
# Display weight distributions for each criterion
print("Individual Weight Statistics:\n")

for name, col in zip(criteria_names, weight_columns):
    weights = consistent_df[col]
    print(f"{name}:")
    print(f"  Mean: {weights.mean():.4f}")
    print(f"  Std:  {weights.std():.4f}")
    print(f"  Min:  {weights.min():.4f}")
    print(f"  Max:  {weights.max():.4f}")
    print()

## 5. Calculate Aggregated Weights

We use **geometric mean** for aggregation, which is the recommended method for AHP.
The geometric mean better preserves the ratio-scale properties of AHP judgments.

In [ ]:
def aggregate_weights_geometric_mean(df, weight_columns):
    """
    Aggregate weights using geometric mean and normalize to sum to 1.
    This is the recommended method for AHP weight aggregation.
    """
    aggregated = {}
    
    for col in weight_columns:
        weights = df[col].values
        # Geometric mean: (product of all values)^(1/n)
        geo_mean = stats.gmean(weights)
        criterion_name = col.replace('weight_', '')
        aggregated[criterion_name] = geo_mean
    
    # Normalize to sum to 1.0
    total = sum(aggregated.values())
    aggregated = {k: v/total for k, v in aggregated.items()}
    
    return aggregated

def aggregate_weights_arithmetic_mean(df, weight_columns):
    """
    Aggregate weights using arithmetic mean (alternative method).
    """
    aggregated = {}
    
    for col in weight_columns:
        weights = df[col].values
        mean = weights.mean()
        criterion_name = col.replace('weight_', '')
        aggregated[criterion_name] = mean
    
    # Normalize to sum to 1.0
    total = sum(aggregated.values())
    aggregated = {k: v/total for k, v in aggregated.items()}
    
    return aggregated

In [ ]:
# Calculate both types for comparison
final_weights_geometric = aggregate_weights_geometric_mean(consistent_df, weight_columns)
final_weights_arithmetic = aggregate_weights_arithmetic_mean(consistent_df, weight_columns)

# Use geometric mean as the primary result (recommended for AHP)
final_weights = final_weights_geometric

print("="*70)
print("FINAL AGGREGATED WEIGHTS (Geometric Mean - RECOMMENDED)")
print("="*70)
print(f"Based on {len(consistent_df)} consistent responses\n")

for criterion, weight in sorted(final_weights.items(), key=lambda x: x[1], reverse=True):
    print(f"{criterion:40s}: {weight:.4f} ({weight*100:.2f}%)")

print(f"\nTotal: {sum(final_weights.values()):.4f}")

print("\n" + "="*70)
print("COMPARISON: Arithmetic Mean")
print("="*70)

for criterion, weight in sorted(final_weights_arithmetic.items(), key=lambda x: x[1], reverse=True):
    print(f"{criterion:40s}: {weight:.4f} ({weight*100:.2f}%)")

## 6. Visualize Results

In [ ]:
import matplotlib.pyplot as plt

# Create figure with subplots
fig, axes = plt.subplots(2, 1, figsize=(12, 10))

# Plot 1: Final weights bar chart
criteria = list(final_weights.keys())
weights = list(final_weights.values())

ax1 = axes[0]
bars = ax1.barh(criteria, weights, color='steelblue')
ax1.set_xlabel('Weight', fontsize=12)
ax1.set_title(f'Final Aggregated Weights (Geometric Mean, n={len(consistent_df)})', fontsize=14, fontweight='bold')
ax1.set_xlim(0, max(weights) * 1.1)

# Add value labels on bars
for bar, weight in zip(bars, weights):
    ax1.text(weight + 0.005, bar.get_y() + bar.get_height()/2, 
             f'{weight:.4f}', va='center', fontsize=10)

# Plot 2: Individual weight distributions (boxplot)
ax2 = axes[1]
weight_data = [consistent_df[col].values for col in weight_columns]

bp = ax2.boxplot(weight_data, labels=criteria_names, vert=False, patch_artist=True)
for patch in bp['boxes']:
    patch.set_facecolor('lightblue')

ax2.set_xlabel('Weight', fontsize=12)
ax2.set_title('Weight Distribution Across Participants', fontsize=14, fontweight='bold')
ax2.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('ahp_weights_visualization.png', dpi=300, bbox_inches='tight')
plt.show()

print("Visualization saved as 'ahp_weights_visualization.png'")

## 7. Export Final Weights

In [ ]:
# Prepare export data
export_data = {
    'metadata': {
        'generated_at': datetime.now().isoformat(),
        'total_responses': len(df),
        'consistent_responses': len(consistent_df),
        'consistency_threshold': CONSISTENCY_THRESHOLD,
        'aggregation_method': 'geometric_mean',
        'criteria_count': len(final_weights)
    },
    'weights': final_weights,
    'weights_percentage': {k: round(v * 100, 2) for k, v in final_weights.items()}
}

# Export to JSON
with open(OUTPUT_JSON, 'w', encoding='utf-8') as f:
    json.dump(export_data, f, indent=2, ensure_ascii=False)

print(f"✓ Weights exported to {OUTPUT_JSON}")

# Export to CSV
weights_df = pd.DataFrame([
    {
        'criterion': k,
        'weight': v,
        'percentage': v * 100
    }
    for k, v in final_weights.items()
])
weights_df = weights_df.sort_values('weight', ascending=False)
weights_df.to_csv(OUTPUT_CSV, index=False)

print(f"✓ Weights exported to {OUTPUT_CSV}")

# Display the exported data
print("\n" + "="*70)
print("EXPORTED DATA PREVIEW")
print("="*70)
print(json.dumps(export_data, indent=2, ensure_ascii=False))

## 8. Generate Summary Report

In [ ]:
# Create a summary report
report = f"""
{'='*70}
AHP WEIGHTS CALCULATOR - SUMMARY REPORT
{'='*70}

Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}

DATA SUMMARY:
  • Total responses collected: {len(df)}
  • Consistent responses (CR ≤ {CONSISTENCY_THRESHOLD}): {len(consistent_df)}
  • Inconsistent responses filtered: {len(df) - len(consistent_df)}
  • Consistency rate: {len(consistent_df)/len(df)*100:.1f}%
  • Number of criteria: {len(final_weights)}

AGGREGATION METHOD:
  Geometric mean (recommended for AHP)

FINAL WEIGHTS (sorted by importance):
"""

for i, (criterion, weight) in enumerate(sorted(final_weights.items(), key=lambda x: x[1], reverse=True), 1):
    report += f"  {i}. {criterion:40s}: {weight:.4f} ({weight*100:.2f}%)\n"

report += f"""
{'='*70}

OUTPUT FILES:
  • {OUTPUT_JSON} - JSON format with metadata
  • {OUTPUT_CSV} - CSV format for spreadsheets
  • ahp_weights_visualization.png - Visual charts

USAGE FOR SCORING:
  Use the weights in '{OUTPUT_JSON}' to calculate weighted scores:
  
  final_score = Σ (criterion_score × criterion_weight)
  
  Example in Python:
    import json
    with open('{OUTPUT_JSON}') as f:
        data = json.load(f)
        weights = data['weights']
    
    # Your scoring logic
    scores = {{'criterion1': 8.5, 'criterion2': 7.2, ...}}
    final_score = sum(scores[c] * weights[c] for c in weights)

{'='*70}
"""

print(report)

# Save report to file
with open('weights_calculation_report.txt', 'w', encoding='utf-8') as f:
    f.write(report)

print("\n✓ Report saved to 'weights_calculation_report.txt'")

## 9. Validate Results

In [ ]:
# Validation checks
print("VALIDATION CHECKS:\n")

# Check 1: Sum of weights = 1.0
weight_sum = sum(final_weights.values())
check1 = abs(weight_sum - 1.0) < 0.0001
print(f"✓ Sum of weights = 1.0: {check1} (sum = {weight_sum:.6f})")

# Check 2: All weights are positive
check2 = all(w > 0 for w in final_weights.values())
print(f"✓ All weights are positive: {check2}")

# Check 3: Number of criteria matches
check3 = len(final_weights) == len(weight_columns)
print(f"✓ Correct number of criteria: {check3} ({len(final_weights)} criteria)")

# Check 4: No NaN or infinite values
check4 = all(np.isfinite(w) for w in final_weights.values())
print(f"✓ No NaN/Inf values: {check4}")

if all([check1, check2, check3, check4]):
    print("\n✅ All validation checks passed!")
else:
    print("\n⚠️  Some validation checks failed. Please review the results.")